# Spacetime FEM — Symbolic Element Matrices (Q1 Bilinear, 1+1D)

This notebook **symbolically** computes the four element-level matrices that arise in a
spacetime finite element discretisation of a 1-D PDE.

The element is a rectangle in the $(x, t)$ plane with side lengths $h_x$ (space) and
$h_t$ (time).  All integrals are evaluated exactly by SymPy — no quadrature is used.

## Matrices computed

| Symbol | Integral | Role |
|--------|----------|------|
| `K_space` | $\int (\partial_x \varphi_i)(\partial_x \varphi_j)\,dx\,dt$ | Spatial diffusion / stiffness |
| `K_mass`  | $\int \varphi_i \varphi_j\,dx\,dt$ | Spacetime mass (reaction) |
| `K_time`  | $\int \varphi_i (\partial_t \varphi_j)\,dx\,dt$ | Temporal advection |
| `K_space_2` | $\int \varphi_i (\partial_x \varphi_j)\,dx\,dt$ | Spatial advection |

These building blocks can be combined (with suitable coefficients) to assemble the global
system for PDEs such as the heat equation, advection–diffusion, or the wave equation.

## 1. Imports and symbolic variables

We declare $x$, $t$, $h_x$, and $h_t$ as positive SymPy symbols so that SymPy can
simplify expressions such as $|h_x| = h_x$ without carrying absolute values.

In [18]:
import sympy as sp

# Spatial coordinate, time coordinate, and element sizes
x, t, hx, ht = sp.symbols('x t hx ht', positive=True)

# Enable pretty-printing for matrix display
sp.init_printing()

## 2. Bilinear (Q1) basis functions

The element occupies $[0, h_x] \times [0, h_t]$.  Nodes are numbered counter-clockwise
starting from the bottom-left corner:

```
4 ──────── 3
│          │   ▲ t
│          │   │
1 ──────── 2   └──▶ x
```

The four **bilinear** shape functions are tensor products of 1-D hat functions:

$$
\varphi_1 = \frac{(h_x - x)(h_t - t)}{h_x h_t}, \qquad
\varphi_2 = \frac{x\,(h_t - t)}{h_x h_t}, \qquad
\varphi_3 = \frac{x\,t}{h_x h_t}, \qquad
\varphi_4 = \frac{(h_x - x)\,t}{h_x h_t}
$$

Each $\varphi_i$ equals 1 at node $i$ and 0 at the other three nodes.

In [ ]:
# Node 1: (0,  0)  —  bottom-left
phi1 = (hx - x) * (ht - t) / (hx * ht)
# Node 2: (hx, 0)  —  bottom-right
phi2 = x * (ht - t) / (hx * ht)
# Node 3: (hx, ht) —  top-right
phi3 = x * t / (hx * ht)
# Node 4: (0,  ht) —  top-left
phi4 = (hx - x) * t / (hx * ht)

phi = [phi1, phi2, phi3, phi4]

# Precompute spatial and temporal derivatives for later use
dphi_dx = [sp.diff(f, x) for f in phi]
dphi_dt = [sp.diff(f, t) for f in phi]

print("Shape functions defined for nodes 1-4.")
for k, f in enumerate(phi, 1):
    print(f"  phi{k} =", f)

Shape functions defined for nodes 1–4.
  phi1 = (ht - t)*(hx - x)/(ht*hx)
  phi2 = x*(ht - t)/(ht*hx)
  phi3 = t*x/(ht*hx)
  phi4 = t*(hx - x)/(ht*hx)


## 3. Spatial stiffness matrix  $K^{\text{space}}$

$$
K^{\text{space}}_{ij} = \int_0^{h_t}\!\int_0^{h_x}
    \frac{\partial \varphi_i}{\partial x}\,
    \frac{\partial \varphi_j}{\partial x}\;dx\,dt
$$

This is the standard Galerkin stiffness matrix for the spatial Laplacian
$-\partial_{xx} u$, integrated over the full spacetime element.
It is **symmetric** by construction.

In [20]:
K_space = sp.Matrix(4, 4, lambda i, j:
    sp.simplify(
        sp.integrate(dphi_dx[i] * dphi_dx[j], (x, 0, hx), (t, 0, ht))
    )
)

print("K_space  (spatial stiffness):")
sp.pprint(K_space)

K_space  (spatial stiffness):
⎡ ht   -ht   -ht    ht ⎤
⎢────  ────  ────  ────⎥
⎢3⋅hx  3⋅hx  6⋅hx  6⋅hx⎥
⎢                      ⎥
⎢-ht    ht    ht   -ht ⎥
⎢────  ────  ────  ────⎥
⎢3⋅hx  3⋅hx  6⋅hx  6⋅hx⎥
⎢                      ⎥
⎢-ht    ht    ht   -ht ⎥
⎢────  ────  ────  ────⎥
⎢6⋅hx  6⋅hx  3⋅hx  3⋅hx⎥
⎢                      ⎥
⎢ ht   -ht   -ht    ht ⎥
⎢────  ────  ────  ────⎥
⎣6⋅hx  6⋅hx  3⋅hx  3⋅hx⎦


## 4. Mass matrix  $K^{\text{mass}}$

$$
K^{\text{mass}}_{ij} = \int_0^{h_t}\!\int_0^{h_x}
    \varphi_i\,\varphi_j\;dx\,dt
$$

The spacetime mass matrix weights the field values themselves (no derivatives).
It is also **symmetric**.


In [21]:
K_mass = sp.Matrix(4, 4, lambda i, j:
    sp.simplify(
        sp.integrate(phi[i] * phi[j], (x, 0, hx), (t, 0, ht))
    )
)

print("K_mass  (spacetime mass):")
sp.pprint(K_mass)

K_mass  (spacetime mass):
⎡ht⋅hx  ht⋅hx  ht⋅hx  ht⋅hx⎤
⎢─────  ─────  ─────  ─────⎥
⎢  9     18     36     18  ⎥
⎢                          ⎥
⎢ht⋅hx  ht⋅hx  ht⋅hx  ht⋅hx⎥
⎢─────  ─────  ─────  ─────⎥
⎢ 18      9     18     36  ⎥
⎢                          ⎥
⎢ht⋅hx  ht⋅hx  ht⋅hx  ht⋅hx⎥
⎢─────  ─────  ─────  ─────⎥
⎢ 36     18      9     18  ⎥
⎢                          ⎥
⎢ht⋅hx  ht⋅hx  ht⋅hx  ht⋅hx⎥
⎢─────  ─────  ─────  ─────⎥
⎣ 18     36     18      9  ⎦


## 5. Temporal advection matrix  $K^{\text{time}}$

$$
K^{\text{time}}_{ij} = \int_0^{h_t}\!\int_0^{h_x}
    \varphi_i\,\frac{\partial \varphi_j}{\partial t}\;dx\,dt
$$

This matrix is **non-symmetric** (the time derivative acts only on the trial function
$\varphi_j$, not on the test function $\varphi_i$).  It captures the first-order
time-derivative term $\partial_t u$ in the PDE.

In [22]:
K_time = sp.Matrix(4, 4, lambda i, j:
    sp.simplify(
        sp.integrate(phi[i] * dphi_dt[j], (x, 0, hx), (t, 0, ht))
    )
)

print("K_time  (temporal advection, phi_i * d/dt phi_j):")
sp.pprint(K_time)

K_time  (temporal advection, phi_i * d/dt phi_j):
⎡-hx   -hx   hx  hx⎤
⎢────  ────  ──  ──⎥
⎢ 6     12   12  6 ⎥
⎢                  ⎥
⎢-hx   -hx   hx  hx⎥
⎢────  ────  ──  ──⎥
⎢ 12    6    6   12⎥
⎢                  ⎥
⎢-hx   -hx   hx  hx⎥
⎢────  ────  ──  ──⎥
⎢ 12    6    6   12⎥
⎢                  ⎥
⎢-hx   -hx   hx  hx⎥
⎢────  ────  ──  ──⎥
⎣ 6     12   12  6 ⎦


## 6. Spatial advection matrix  $K^{\text{space\_2}}$

$$
K^{\text{space\_2}}_{ij} = \int_0^{h_t}\!\int_0^{h_x}
    \varphi_i\,\frac{\partial \varphi_j}{\partial x}\;dx\,dt
$$

Like `K_time`, this matrix is **non-symmetric**.  The spatial derivative acts only on the
trial function.  It is used in the weak form of advection terms $c\,\partial_x u$ when the
advection speed $c$ is absorbed into the test-function side.

In [23]:
K_space_2 = sp.Matrix(4, 4, lambda i, j:
    sp.simplify(
        sp.integrate(phi[i] * dphi_dx[j], (x, 0, hx), (t, 0, ht))
    )
)

print("K_space_2  (spatial advection, phi_i * d/dx phi_j):")
sp.pprint(K_space_2)

K_space_2  (spatial advection, phi_i * d/dx phi_j):
⎡-ht   ht  ht  -ht ⎤
⎢────  ──  ──  ────⎥
⎢ 6    6   12   12 ⎥
⎢                  ⎥
⎢-ht   ht  ht  -ht ⎥
⎢────  ──  ──  ────⎥
⎢ 6    6   12   12 ⎥
⎢                  ⎥
⎢-ht   ht  ht  -ht ⎥
⎢────  ──  ──  ────⎥
⎢ 12   12  6    6  ⎥
⎢                  ⎥
⎢-ht   ht  ht  -ht ⎥
⎢────  ──  ──  ────⎥
⎣ 12   12  6    6  ⎦


## 7. Summary and usage

The four matrices above are the **element-level building blocks** for spacetime FEM.
To form the element stiffness matrix for a specific PDE, combine them as follows:

| PDE | Element matrix |
|-----|---------------|
| Heat equation $\partial_t u - \kappa\,\partial_{xx} u = f$ | $K^{\text{time}} - \kappa\,K^{\text{space}}$ |
| Advection equation $\partial_t u + c\,\partial_x u = f$ | $K^{\text{time}} + c\,K^{\text{space\_2}}$ |
| Advection–diffusion $\partial_t u + c\,\partial_x u - \kappa\,\partial_{xx} u = f$ | $K^{\text{time}} + c\,K^{\text{space\_2}} - \kappa\,K^{\text{space}}$ |
| Reaction–diffusion $\partial_t u + \mu\,u - \kappa\,\partial_{xx} u = f$ | $K^{\text{time}} + \mu\,K^{\text{mass}} - \kappa\,K^{\text{space}}$ |

These element matrices are then **assembled** into the global system by summing
contributions from all elements, with appropriate handling of shared degrees of freedom
at element boundaries.